In [5]:
import pandas as pd

In [4]:
validation_files={
    'ewc':"/Users/sefika/phd_projects/llm-catastrophic-re/results_all/fewrel_corrected_results/tmlr/ewc/canonical_errors/validation_results.json",
    'mas':"/Users/sefika/phd_projects/llm-catastrophic-re/results_all/fewrel_corrected_results/tmlr/mas/canonical_errors/validation_results.json",
    'baseline':"/Users/sefika/phd_projects/llm-catastrophic-re/results_all/fewrel_corrected_results/tmlr/baseline/canonical_errors/validation_results.json",
    'si':"/Users/sefika/phd_projects/llm-catastrophic-re/results_all/fewrel_corrected_results/tmlr/si/canonical_errors/validation_results.json"   
}

In [6]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score


In [1]:
import sentence_transformers as sbert
from sklearn.metrics.pairwise import cosine_similarity
model = sbert.SentenceTransformer('all-MiniLM-L6-v2')

/opt/anaconda3/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


README.md: 0.00B [00:00, ?B/s]

/opt/anaconda3/lib/python3.11/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [11]:

def bert_sim_metric(df_ewc, message_type='Inverse'):
    
    results = []

    for run_id in range(1,2):
        df_run = df_ewc[df_ewc['run_id']==run_id]
        task_ids = df_run['task_id'].unique()
        for task_id in range(8,9):
            task_relations = df_run[df_run['task_id']==task_id]
            gt_relations = []
            pred_relations = []
            for inverse_index, inverse_row in task_relations.iterrows():
                gt_relation = inverse_row['gt_relation']
                
                pred =  inverse_row['prediction']
                pred_encode = model.encode([pred])
                gt_encode = model.encode([gt_relation])
                sim_score = cosine_similarity(gt_encode, pred_encode).item()
                if sim_score > 0.7:
                    pred_relations.append(gt_relation)
                else:
                    pred_relations.append(pred)
                gt_relations.append(gt_relation)
            acc = accuracy_score(gt_relations, pred_relations)
            print(f"Run ID: {run_id}, Task ID: {task_id}, BERT Similarity Accuracy: {acc}")
            results.append({
                'run_id': run_id,
                'task_id': task_id,
                'bert_similarity': acc
            })
    return results


In [12]:
df_pred = pd.json_normalize(pd.read_json(validation_files['baseline']).to_dict(orient="records"))
results = bert_sim_metric(df_pred, message_type='Canonical')
results_df = pd.DataFrame(results)
results_df.groupby('task_id').mean().T


KeyboardInterrupt: 

In [26]:
df_pred = pd.json_normalize(pd.read_json(validation_files['ewc']).to_dict(orient="records"))
results = inverse_analysis(df_pred, message_type='Canonical')
results_df = pd.DataFrame(results)
results_df.groupby('task_id').mean().T

task_id,1,2,3,4,5,6,7,8
run_id,3.000,3.000000,3.000000,3.00000,3.000000,3.000000,3.000000,3.000000
accuracy,0.981,0.907571,0.872476,0.86825,0.859057,0.858262,0.855306,0.856625


In [27]:
df_pred = pd.json_normalize(pd.read_json(validation_files['mas']).to_dict(orient="records"))
results = inverse_analysis(df_pred, message_type='Canonical')
results_df = pd.DataFrame(results)
results_df.groupby('task_id').mean().T

task_id,1,2,3,4,5,6,7,8
run_id,3.000000,3.000000,3.00000,3.000000,3.000000,3.000000,3.000000,3.000000
accuracy,0.980571,0.910429,0.87281,0.869643,0.858057,0.856429,0.854878,0.856089


In [28]:
df_pred = pd.json_normalize(pd.read_json(validation_files['si']).to_dict(orient="records"))
results = inverse_analysis(df_pred, message_type='Canonical')
results_df = pd.DataFrame(results)
results_df.groupby('task_id').mean().T

task_id,1,2,3,4,5,6,7,8
run_id,3.000000,3.000000,3.00000,3.000000,3.000000,3.000000,3.000000,3.000000
accuracy,0.977571,0.911429,0.88319,0.873464,0.861857,0.865476,0.864388,0.860768
